# Foraging: 3 models x 2 learning approaches

One grid, six runs. Every cell trains on the **same task, same reward, same seed** --
only the model and the training algorithm change, so the two axes can be read
independently.

## The three models (how much structure is given, before any learning)

| model | `network` | `controller` | given for free | must be learned |
|---|---|---|---|---|
| `mlp` | `'mlp'` | -- | nothing | swimming **and** steering |
| `ncap` | `'ncap'` | `None` | the C. elegans swimming circuit | steering -- but there is **no turn input to learn** |
| `ncap_steer` | `'ncap'` | `'learned_steering'` | the circuit **and** a turn primitive | only *which way* to turn |

`ncap` is the honest middle rung, and its constraint is structural rather than a matter
of budget: with `controller=None` the circuit is built `include_turn_control=False`, so
there is no turn input for any optimizer to drive. Whatever it scores, it scores without
steering.

## The three learning approaches

| method | family | what it optimizes |
|---|---|---|
| `ppo` | on-policy RL (tonic) | gradient on a stochastic policy |
| `es` | Evolution Strategies | rank-transformed finite differences, no gradient |

(`ddpg` is supported and slots straight into `METHODS`, but is left out here: it costs
~5x PPO for no extra insight on this task.)

ES is how the NCAP paper itself trains the circuit, which is why it earns a column
rather than being treated as an also-ran.

## Reading the results

**The success bars are the metric of record, not the reward curves.** `success_rate` is
physics-only: the fraction of fresh episodes whose head actually reaches the food, read
from simulator state and independent of whatever reward trained the checkpoint. Aggregate
reward has repeatedly lied in this environment -- a "great" score once turned out to be
almost entirely swim speed with no food-seeking under it, and ES in particular banks
`progress_reward` for drifting toward food without ever closing on it.

Each model is also scored **untrained**, so training's contribution and the model's
contribution never get confused for each other.

In [ ]:
# --- Setup -------------------------------------------------------------------------
%matplotlib inline
import sys
from pathlib import Path

# This notebook only orchestrates; every function/class lives in macrocircuits under src/.
SRC = str(Path.cwd().parent / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from macrocircuits import ensure_tonic
ensure_tonic()  # clone neuromatch/tonic next to this notebook (once) and put it on the path

import torch, tonic, tonic.torch
from dm_control.mujoco import engine
from IPython.display import Video

from macrocircuits.es import es_config, is_es_trained, make_es_policy, run_es
from macrocircuits.training import resolve_runs, run_config, run_path, is_trained, train
from macrocircuits.video import write_video

# --- Knobs -------------------------------------------------------------------------
N_JOINTS      = 5          # 6-link swimmer
SUCCESS_DIST  = 0.15       # head within this of the food during an episode => it found it
EVAL_SEED     = 0          # which fresh episodes are drawn; numbers shift ~+/-10% across seeds
EVAL_EPISODES = 30

# Training budgets -- the main knobs for a longer/shorter run.
STEPS_RL       = int(1e6)  # PPO env steps -- the paper's budget (~22-29 min each)
ES_GENERATIONS = 100       # ES generations; population 64, ~45 s/gen => ~1.2 h each

# foraging ships rewardless (every term defaults to 0.0 -- see envs.foraging), so a run
# with no task_kwargs has nothing to learn from. Same dense shaping as the sibling
# notebooks, which keeps all these numbers on one axis.
FORAGE_REWARD = dict(progress_reward_weight=10.0, eat_bonus=5.0)

# Categorical palette, one hue per learning approach, validated for CVD separation and
# for contrast against a light surface (every bar also carries a direct value label,
# so identity never rests on colour alone). Grey is the untrained reference, not a series.
UNTRAINED_COLOR = '#8a8a83'
METHOD_COLOR = {'ppo': '#2a78d6', 'ddpg': '#eb6834', 'es': '#1baf7a'}

print('ready')

## Helpers

Physics-only evaluation and rendering, shared by all nine runs. Three wrinkles the grid
forces that a single-model notebook never hits:

- **Checkpoints load two different ways.** A tonic RL run (`ppo`/`ddpg`) is rebuilt from
  its `config.yaml` as a full agent; an ES run is a bare policy `state_dict`. Both come
  back as the same `(policy, env, physics)` triple, so one `success_rate` scores all nine.
- **The eval observation differs by model.** NCAP's RL actor reads a timestep appended by
  tonic (`time_feature=True`); the MLP baseline does not. `trained_policy` sidesteps this
  by rebuilding the env from the run's own config. ES policies never read a time feature
  on either network, so they get the plain env.
- **ES + NCAP needs `reset()` between episodes.** `NCAPSwimmerPolicy` drives its head
  oscillator from an *internal step counter* rather than from the observation, so without
  a reset every episode after the first starts mid-phase. The RL actors read phase from
  the time feature and have no `reset`, hence the `hasattr` guard.

In [ ]:
def _get_physics(env):
    while env is not None:
        if hasattr(env, 'physics'):
            return env.physics
        inner = getattr(env, 'environment', None)
        if inner is not None and hasattr(inner, 'physics'):
            return inner.physics
        env = getattr(env, 'env', None)
    raise RuntimeError('no physics in env chain')


def _arena_frame(physics, distance=5.5):
    """Fixed top-down camera on the whole arena, so you see the worm traverse to food.

    Deliberately not one of the model's own cameras: 0 and 1 track the worm's position
    but never rotate with its heading, so a turn can read as the wrong way round purely
    from the camera. Nothing rotates here, so direction on screen is real.
    """
    cam = engine.Camera(physics, height=480, width=640, camera_id=-1)
    rc = cam._render_camera
    rc.lookat[:] = [0.0, 0.0, 0.0]
    rc.distance = distance; rc.azimuth = 90; rc.elevation = -90
    frame = cam.render(); cam._scene.free()
    return frame


def _bare_eval_env(seed=EVAL_SEED):
    """Foraging env with NO time feature -- what an ES policy (either network) sees.
    The reward shaping is irrelevant here: success_rate is physics-only, so the env's
    reward is never read."""
    env = tonic.environments.distribute(
        lambda: tonic.environments.ControlSuite('swimmer-foraging'))
    env.initialize(seed)
    return env


def _wrap(policy, env):
    """Torch policy -> the plain (obs -> action) callable success_rate expects, carrying
    `reset` through when the policy has one (see the ES/NCAP oscillator note above)."""
    policy.eval()
    def policy_fn(obs):
        with torch.no_grad():
            return policy(torch.as_tensor(obs, dtype=torch.float32)).numpy()
    if hasattr(policy, 'reset'):
        policy_fn.reset = policy.reset
    return policy_fn, env, _get_physics(env.environments[0])


def _es_sizes(env):
    """The two size attributes make_es_policy reads off a training agent."""
    from types import SimpleNamespace
    return SimpleNamespace(observation_size=env.observation_space.shape[-1],
                           action_size=env.action_space.shape[-1])


def trained_policy(path, seed=EVAL_SEED):
    """Load a tonic RL checkpoint (PPO or DDPG); returns (policy, env, physics).
    Rebuilds the exact agent and env from the run's config.yaml, then the last step_*."""
    import argparse, yaml
    import macrocircuits.training as _t
    ns = dict(vars(_t))
    cfg = argparse.Namespace(
        **yaml.load(open(os.path.join(path, 'config.yaml')), Loader=yaml.FullLoader))
    try:
        if cfg.header: exec(cfg.header, ns)
    except Exception:
        pass
    agent = eval(cfg.agent, ns)
    env = tonic.environments.distribute(lambda: eval(cfg.environment, ns))
    env.initialize(seed)
    agent.initialize(observation_space=env.observation_space,
                     action_space=env.action_space, seed=seed)
    cp = os.path.join(path, 'checkpoints')
    ids = [int(n.split('.')[0][5:]) for n in os.listdir(cp) if n.startswith('step_')]
    agent.load(os.path.join(cp, f'step_{max(ids)}'))
    return (lambda obs: agent.test_step(obs, 0)), env, _get_physics(env.environments[0])


def es_policy(path, seed=EVAL_SEED):
    """Load an ES checkpoint (either network); returns (policy, env, physics).
    An ES checkpoint is a bare state_dict, so the policy is rebuilt from config.yaml --
    including its controller, which is why the saved controller/controller_kwargs matter."""
    import yaml
    with open(os.path.join(path, 'config.yaml')) as f:
        cfg = yaml.safe_load(f)
    env = _bare_eval_env(seed)
    policy = make_es_policy(
        cfg['network'], _es_sizes(env),
        hidden_sizes=tuple(cfg.get('hidden_sizes') or (64, 64)),
        swimmer_kwargs=cfg.get('swimmer_kwargs'),
        controller=cfg.get('controller'),
        controller_kwargs=cfg.get('controller_kwargs'),
    )
    policy.load_state_dict(
        torch.load(os.path.join(path, 'checkpoints', 'best.pt'), map_location='cpu'))
    return _wrap(policy, env)


def untrained_policy(model, seed=EVAL_SEED):
    """The model before ANY learning -- the floor its trained runs have to beat.

    Built as a bare ES-style policy for all three models so the floors are measured on
    one footing (no tonic agent, no time feature). For `ncap_steer` this still includes
    the controller's behaviour-cloned warm start, which is part of the model, not of
    training -- from random init PPO never even found the correct turn sign."""
    torch.manual_seed(seed)
    env = _bare_eval_env(seed)
    policy = make_es_policy('mlp' if model == 'mlp' else 'ncap', _es_sizes(env),
                            controller=MODELS[model]['controller'])
    return _wrap(policy, env)


def load_policy(run, seed=EVAL_SEED):
    """Dispatch on method: ES checkpoints load differently from tonic RL ones."""
    path = run_path(**run)
    return es_policy(path, seed) if run['method'] == 'es' else trained_policy(path, seed)


def success_rate(policy, env, n_episodes=EVAL_EPISODES):
    """Fraction of fresh episodes whose head comes within SUCCESS_DIST of the food.

    Distance is read from infos['observations'] -- the *pre-reset* transition
    observation. Reading it from the env after a reset instead measures the NEXT
    episode's starting position, a measurement bug that once produced a whole
    afternoon of confidently wrong conclusions."""
    tgt = slice(N_JOINTS, N_JOINTS + 2)
    if hasattr(policy, 'reset'):
        policy.reset()
    obs = env.start(); mind = np.inf; hits = []
    while len(hits) < n_episodes:
        obs, infos = env.step(policy(obs))
        mind = min(mind, float(np.linalg.norm(infos['observations'][0, tgt])))
        if infos['resets'][0]:
            hits.append(mind < SUCCESS_DIST); mind = np.inf
            if hasattr(policy, 'reset'):
                policy.reset()
    return float(np.mean(hits))


def forage_episode(policy, env, physics, steps=1000, viz_food_size=0.10):
    """One episode; records overhead frames, head path, food positions, and the steps a
    food was reached (it then respawns). viz_food_size only enlarges the food marker for
    the video -- the success metric uses the true tiny food."""
    if hasattr(policy, 'reset'):
        policy.reset()
    obs = env.start()
    if viz_food_size:
        physics.named.model.geom_size['target', 0] = viz_food_size; physics.forward()
    frames, hx, hy, fx, fy, eats = [], [], [], [], [], []
    prev_food = None
    for i in range(steps):
        frames.append(_arena_frame(physics))
        nose = physics.named.data.geom_xpos['nose'][:2].copy()
        food = physics.named.data.geom_xpos['target'][:2].copy()
        hx.append(nose[0]); hy.append(nose[1]); fx.append(food[0]); fy.append(food[1])
        if prev_food is not None and np.linalg.norm(food - prev_food) > 1e-6:
            eats.append(i)
        prev_food = food
        obs, infos = env.step(policy(obs))
        if infos['resets'][0]:
            break
    return dict(frames=frames, hx=np.array(hx), hy=np.array(hy),
                fx=np.array(fx), fy=np.array(fy), eats=eats)


def show_video(frames, path, fps=30):
    """Write frames to mp4 and embed directly (plays reliably in the VSCode viewer)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    write_video(path, frames, fps=fps)
    return Video(path, embed=True, width=560)


def plot_path(ep, title):
    plt.figure(figsize=(6.5, 6.5))
    plt.scatter(ep['hx'], ep['hy'], c=np.arange(len(ep['hx'])), cmap='viridis', s=6,
                label='head path')
    food_pts = np.unique(np.c_[ep['fx'], ep['fy']], axis=0)
    plt.scatter(food_pts[:, 0], food_pts[:, 1], marker='*', s=260, c='red', ec='k',
                label='food', zorder=5)
    if ep['eats']:
        plt.scatter(ep['hx'][ep['eats']], ep['hy'][ep['eats']], s=90, c='lime', ec='k',
                    label='reached', zorder=6)
    plt.gca().set_aspect('equal'); plt.grid(alpha=0.3); plt.legend()
    plt.title(title); plt.xlabel('world x'); plt.ylabel('world y'); plt.show()

print('helpers ready')

## The grid

Nine runs from three model specs x three methods. Everything else -- task, reward, seed --
comes from one shared `DEFAULTS`, so the grid is the only thing that varies.

Labels follow the existing `<model>_<method>_forage_poc` convention, which means the three
`mlp_*` cells **reuse the checkpoints the sibling `forage_mlp_baseline_poc.ipynb` already
trained** rather than retraining them. `is_trained` / `is_es_trained` compare the full
config, so if you change `STEPS_RL`, `ES_GENERATIONS` or `FORAGE_REWARD` they stop
matching and retrain -- which is the correct behaviour, not a bug.

In [ ]:
MODELS = {
    'mlp':        dict(network='mlp',  controller=None),               # nothing given
    'ncap':       dict(network='ncap', controller=None),               # gait, no steering
    'ncap_steer': dict(network='ncap', controller='learned_steering'), # gait + steering
}
# 'ddpg' is implemented and works, but was dropped from the overnight grid: measured
# at 677 s per 1e5 steps it is ~5x PPO's cost for no extra insight here. Re-add it to
# this tuple to get the full 3x3 back.
METHODS = ('ppo', 'es')

DEFAULTS = dict(task='foraging', task_kwargs=FORAGE_REWARD, seed=EVAL_SEED)

# steps is RL-only and generations is ES-only; resolve_runs rejects a run that sets the
# knob its own method would ignore, which is why the budget is attached per method.
_specs, _keys = [], []
for model, model_cfg in MODELS.items():
    for method in METHODS:
        spec = dict(model_cfg, method=method, label=f'{model}_{method}_forage_poc')
        spec.update({'generations': ES_GENERATIONS} if method == 'es'
                    else {'steps': STEPS_RL})
        _specs.append(spec); _keys.append((model, method))

RUNS = dict(zip(_keys, resolve_runs(_specs, defaults=DEFAULTS)))

print(f'{"model":<11} {"method":<5} {"budget":>21}   {"trained?":<9} path')
for (model, method), run in RUNS.items():
    path = run_path(**run)
    if method == 'es':
        done = is_es_trained(path, **es_config(**run))
        budget = f'{run["generations"]:>7,} generations'
    else:
        done = is_trained(path, *run_config(**run)[:2], run_config(**run)[3])
        budget = f'{run["steps"]:>7,} steps      '
    print(f'{model:<11} {method:<5} {budget}   {"yes" if done else "NO":<9} {path}')

## Train

Trains every run that is not already checkpointed with these exact settings.

> **Cost (measured, not guessed).** These are *full* budgets, not smoke tests:
> 1e6 steps is the paper's RL budget and 100 generations is 6.4M env steps of ES.
> PPO ~22-29 min per run; ES ~45 s/generation => ~1.2 h per run. **Six runs ~ 5 h**
> -- an overnight job. Note that changing any budget makes every cached run stop
> matching, so all six retrain together.


In [ ]:
import time, traceback

# (model, method) -> outcome string. Kept so the summary below, and every cell after it,
# can tell "never trained" apart from "trained fine".
TRAIN_REPORT = {}
start = time.time()

for (model, method), run in RUNS.items():
    label, path = run['label'], run_path(**run)
    t = time.time()
    try:
        if method == 'es':
            if is_es_trained(path, **es_config(**run)):
                TRAIN_REPORT[(model, method)] = 'skipped (already trained)'
                print(f'[skip]  {label}')
                continue
            print()
            print(f'[start] {label}  ({run["generations"]:,} generations)')
            run_es(**es_config(**run))
        else:
            agent, environment, name, trainer = run_config(**run)
            if is_trained(path, agent, environment, trainer):
                TRAIN_REPORT[(model, method)] = 'skipped (already trained)'
                print(f'[skip]  {label}')
                continue
            print()
            print(f'[start] {label}  ({run["steps"]:,} steps)')
            train('import tonic.torch', agent, environment, name=name, trainer=trainer)
        TRAIN_REPORT[(model, method)] = f'trained in {(time.time() - t) / 60:.1f} min'
        print(f'[done]  {label} in {(time.time() - t) / 60:.1f} min')
    # `except Exception` deliberately does NOT catch KeyboardInterrupt (a BaseException),
    # so Interrupt Kernel still stops the whole grid rather than skipping to the next run.
    except Exception as exc:
        TRAIN_REPORT[(model, method)] = f'FAILED: {type(exc).__name__}: {exc}'
        print(f'[FAIL]  {label} after {(time.time() - t) / 60:.1f} min -- '
              f'continuing with the rest of the grid')
        traceback.print_exc()

print()
print(f'=== finished in {(time.time() - start) / 60:.1f} min ===')
for key, outcome in TRAIN_REPORT.items():
    mark = 'x' if outcome.startswith('FAILED') else '.'
    print(f'  [{mark}] {key[0]:<11} {key[1]:<5} {outcome}')

_failed = [k for k, v in TRAIN_REPORT.items() if v.startswith('FAILED')]
if _failed:
    print()
    print(f'{len(_failed)} run(s) failed: {_failed}')
    print('Every cell below skips runs with no checkpoint, so the rest of the notebook')
    print('still works -- a failed cell just shows "--" in the success table.')

## Learning curves

One panel per model, one line per learning approach, read from each run's `log.csv`.

> These are **reward** curves: read them as "did this optimizer climb anything", not as
> "did it forage". Two caveats make them non-comparable across methods even so -- the
> tonic runs log the mean test-episode score while ES logs its running *best* return, and
> the x-axes count different things (ES charges every population member, so its sample
> cost is large and honest). The success bars below are the head-to-head comparison.

In [ ]:
def curve(run):
    """(x, y) for one run's log.csv, handling tonic's and ES's different schemas."""
    log = pd.read_csv(os.path.join(run_path(**run), 'log.csv'))
    ycol = next(c for c in log.columns if 'episode_score/mean' in c)
    # ES writes its own columns; tonic's cumulative counter is the one ending '/steps'
    # (NOT 'train/epoch_steps', which is the constant epoch length and is identical at
    # every eval point -- plotting against it collapses the curve onto a single x).
    xcol = ('total_env_steps' if 'total_env_steps' in log.columns
            else next(c for c in log.columns if c.endswith('/steps')))
    return log[xcol], log[ycol]


fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, model in zip(axes, MODELS):
    for method in METHODS:
        try:
            x, y = curve(RUNS[(model, method)])
        except (FileNotFoundError, StopIteration):
            continue  # not trained yet
        ax.plot(x, y, marker='o', ms=4, lw=2, color=METHOD_COLOR[method], label=method)
    ax.set_title(model); ax.set_xlabel('env steps'); ax.grid(alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('test episode score (mean)')
axes[0].legend(frameon=False)
fig.suptitle('training reward per model -- NOT a foraging measure', y=1.03)
plt.tight_layout(); plt.show()

## Physics-only success -- the money plot

The fraction of `EVAL_EPISODES` fresh episodes whose head reaches the food. Each model's
**untrained** floor is the grey bar, so the height of the coloured bars above it is what
that learning approach actually bought on that model.

In [ ]:
SUCCESS = {}
for model in MODELS:
    SUCCESS[(model, 'untrained')] = success_rate(*untrained_policy(model)[:2])
    for method in METHODS:
        run = RUNS[(model, method)]
        try:
            pol, env, _ = load_policy(run)
        except (FileNotFoundError, IndexError, ValueError):
            print(f'skip {run["label"]}: no checkpoint yet')
            continue
        SUCCESS[(model, method)] = success_rate(pol, env)

# Table view: the numbers themselves, never only encoded as bar height.
cols = ['untrained', *METHODS]
table = pd.DataFrame(
    [[SUCCESS.get((m, c), np.nan) * 100 for c in cols] for m in MODELS],
    index=list(MODELS), columns=cols,
)
print('physics-only success (%), '
      f'{EVAL_EPISODES} fresh episodes, eval seed {EVAL_SEED}\n')
print(table.round(0).to_string(na_rep='  --'))

In [ ]:
series = [('untrained', UNTRAINED_COLOR), *((m, METHOD_COLOR[m]) for m in METHODS)]
x = np.arange(len(MODELS))
width = 0.2

fig, ax = plt.subplots(figsize=(8.5, 4.6))
for i, (name, color) in enumerate(series):
    offset = (i - (len(series) - 1) / 2) * width
    vals = [SUCCESS.get((model, name), np.nan) * 100 for model in MODELS]
    # linewidth draws the 2px surface gap that keeps adjacent bars from fusing.
    bars = ax.bar(x + offset, vals, width * 0.92, label=name, color=color,
                  edgecolor='white', linewidth=2, zorder=3)
    ax.bar_label(bars, fmt='%.0f', padding=2, fontsize=9, color='#3d3d38')

ax.set_xticks(x, list(MODELS))
ax.set_ylabel('success (%)'); ax.set_ylim(0, 100)
ax.set_title(f'reaching food, {EVAL_EPISODES} fresh episodes (eval seed {EVAL_SEED})')
ax.legend(frameon=False, ncol=len(series), loc='upper left')
ax.grid(alpha=0.25, axis='y', zorder=0)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## Watch the best cell in the grid

Renders one episode from whichever (model, method) scored highest, on a fresh seed, with
its head path and the food it reached.

In [ ]:
best_key = max((k for k in SUCCESS if k[1] != 'untrained'), key=SUCCESS.get)
best_run = RUNS[best_key]
print(f'best cell: {best_key[0]} x {best_key[1]} '
      f'({SUCCESS[best_key] * 100:.0f}%) -> {best_run["label"]}')

pol, env, phys = load_policy(best_run, seed=3)
ep = forage_episode(pol, env, phys, steps=1000)
print(f'food reached this episode: {len(ep["eats"])}')
show_video(ep['frames'][::2], f'output_videos/forage_{best_run["label"]}.mp4', fps=30)

In [ ]:
plot_path(ep, f'{best_run["label"]}: head path and food')